### Модели AR и MA

| **Модели AR** | **Модели MA** |
|:-------------|:---------------|
|Зависят от прошлых значений ряда. | Зависят от прошлых ошибок прогноза. |
| Подходит, когда прошлые значения оказывают прямое влияние на будущие значения и для медленно меняющихся временных рядов | Полезно, когда ряд лучше объясняется шоками или случайными возмущениями, т. е. временными рядами с внезапными изменениями|
| Если **PACF** резко падает при заданном лаге $p$, то используйте модель **AR** с порядком $p$| Если **ACF** резко падает при заданном лаге $q$, то используйте модель **MA** с порядком $q$ |

В этой лекции вы должны были изучить основы:

1. Автокорреляционная функция (ACF).
2. Частичная автокорреляционная функция (PACF).
3. Авторегрессионные (AR) модели.
4. Выбор порядка $p$.
5. Модели скользящего среднего (MA).
6. Выбор порядка $q$.
7. Объединение сглаживателей для прогнозирования тренда и сезонности с моделями AR/MA для прогнозирования остатков.

## Упражнения

- Загрузите два временных ряда `arma_ts1` и `arma_ts2`, выполнив код ниже.

In [10]:
import numpy as np
import requests
from io import BytesIO
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error as mse

# Load the first time series
response = requests.get("https://zenodo.org/records/10951538/files/arma_ts3.npz?download=1")
response.raise_for_status()
arma_ts1 = np.load(BytesIO(response.content))['signal']
print(len(arma_ts1))

# Load the second time series
response = requests.get("https://zenodo.org/records/10951538/files/arma_ts4.npz?download=1")
response.raise_for_status()
arma_ts2 = np.load(BytesIO(response.content))['signal']
print(len(arma_ts2))

# 1. разделение на трейн и тест
train1, test1 = arma_ts1[:-30], arma_ts1[-30:]
train2, test2 = arma_ts2[:-100], arma_ts2[-100:]


# 2. стационарность через разности
d1 = np.diff(train1)
d2 = np.diff(train2)

# p=1 по pacf и q=2 по acf
p1, q1 = 2, 1
m_ar1 = ARIMA(train1, order=(p1, 1, 0)).fit()
f_ar1 = m_ar1.forecast(30)
m_ma1 = ARIMA(train1, order=(0, 1, q1)).fit()
f_ma1 = m_ma1.forecast(30)

print(f"ряд 1 p={p1} q={q1}")
print(f"прогноз ar1: {f_ar1[:5]}")
print(f"прогноз ma1: {f_ma1[:5]}")

# для второго ряда p=1 и q=2
p2, q2 = 1, 2
m_ar2 = ARIMA(train2, order=(p2, 1, 0)).fit()
f_ar2 = m_ar2.forecast(100)
m_ma2 = ARIMA(train2, order=(0, 1, q2)).fit()
f_ma2 = m_ma2.forecast(100)

print(f"ряд 2 p={p2} q={q2}")
print(f"прогноз ar2: {f_ar2[:5]}")
print(f"прогноз ma2: {f_ma2[:5]}")

# для первого ряда
print("ряд 1 mse ar:", mse(test1, f_ar1))
print("ряд 1 mse ma:", mse(test1, f_ma1))

# для второго ряда
print("ряд 2 mse ar:", mse(test2, f_ar2))
print("ряд 2 mse ma:", mse(test2, f_ma2))

479
1000
ряд 1 p=2 q=1
прогноз ar1: [-39.42927801 -39.2140993  -39.11618055 -39.16703169 -39.17174286]
прогноз ma1: [-38.6134513 -38.6134513 -38.6134513 -38.6134513 -38.6134513]
ряд 2 p=1 q=2
прогноз ar2: [43.68372694 44.69445224 44.02125799 44.46963946 44.17099469]
прогноз ma2: [43.4872246  43.80759369 43.80759369 43.80759369 43.80759369]
ряд 1 mse ar: 79.34126121739163
ряд 1 mse ma: 88.62346752094278
ряд 2 mse ar: 14.005052998359634
ряд 2 mse ma: 17.26277219330291


Для каждого временного ряда:

1. Разделите временной ряд на обучающий и тестовый.
    - Используйте последние 30 значений в качестве тестовых для первого временного ряда.
    - Используйте последние 100 значений в качестве тестовых для второго временного ряда.
2. Сделайте временной ряд стационарным.
3. Определите порядок $p$ модели AR.
4. Вычислите прогноз тестовых данных с помощью модели AR($p$).
5. Определите порядок $q$ модели MA.
6. Вычислите прогноз тестовых данных с помощью модели MA($q$).